# 2. SAP RPT 1.6 on a synthetic invoice dataset

## Goal

Validate the shared masked request, review the saved direct BTP result, and optionally send a new inference request. The main dataset has 187 fraud-labelled context rows and 22 fraud-labelled test rows. Saved results and reduced provenance are included so the notebook is readable without SAP credentials.

For a new live run, copy `.env.example` to `.env` and enter your SAP AI Core service-key settings. Keep credentials out of notebook cells and commits. A live request can incur SAP inference charges.

## 1. Validate the request and shared split

The first 1,024 rows provide labeled context. Both targets in the final 128 rows must be `[PREDICT]`, so their true labels are not sent for inference.

In [1]:
import hashlib
import json
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown
from sklearn.metrics import accuracy_score, precision_score, recall_score

ROOT = next(path for path in (Path.cwd(), Path.cwd().parent) if (path / "data/synthetic_invoices/payment_behavior.csv").exists())
sys.path.insert(0, str(ROOT / "scripts"))
from generate_synthetic_invoices import CONTEXT_ROWS, PREDICTION_ROWS, TARGETS
DATA = ROOT / "data/synthetic_invoices"

source = pd.read_csv(DATA / "payment_behavior.csv")
request_path = DATA / "rpt16_request.json"
request = json.loads(request_path.read_text(encoding="utf-8"))
request_rows = pd.DataFrame(request["rows"])
assert len(source) == len(request_rows) == CONTEXT_ROWS + PREDICTION_ROWS
assert request["index_column"] == "invoice_id"
assert {column["name"] for column in request["prediction_config"]["target_columns"]} == set(TARGETS)
assert request_rows.invoice_id.tolist() == source.invoice_id.tolist()
feature_columns = [column for column in source.columns if column not in TARGETS]
pd.testing.assert_frame_equal(request_rows[feature_columns], source[feature_columns], check_dtype=False)
for target in TARGETS:
    assert request_rows[target].iloc[:CONTEXT_ROWS].astype(int).tolist() == source[target].iloc[:CONTEXT_ROWS].tolist()
    assert request_rows[target].iloc[CONTEXT_ROWS:].eq("[PREDICT]").all()
print(f"Validated {CONTEXT_ROWS} context and {PREDICTION_ROWS} masked test rows.")
print(f"Request SHA-256: {hashlib.sha256(request_path.read_bytes()).hexdigest()}")

Validated 1024 context and 128 masked test rows.
Request SHA-256: ee25a1f418c6e728fb5ceccef927a3246c6c8c6bf527a77d14a743a4d07fa54f


## 2. Review or rerun RPT through BTP

The published notebook reviews the saved BTP result by default. Set `RUN_LIVE_RPT = True` only when you intend to send a new request; each execution then calls the deployment and updates the saved result. Python booleans use `True` and `False`.

In [2]:
RUN_LIVE_RPT = False  # Set to True only for a new live BTP request

if RUN_LIVE_RPT:
    from run_rpt16_invoices import has_rpt_configuration, load_local_env, run_experiment
    load_local_env()
    if not has_rpt_configuration():
        raise RuntimeError("Configure direct RPT credentials or SAP AI Core service-key settings in .env")
    result_path = run_experiment()
    print(f"Saved {result_path.relative_to(ROOT)}")
else:
    display(Markdown("**Live BTP call skipped.** Review the saved results in section 3 below."))

**Live BTP call skipped.** Review the saved results in section 3 below.

## 3. Review held-out BTP results

This uses the saved BTP response and makes no inference request. ROC AUC is shown as N/A because the top-1 response does not provide a score for the positive class on every invoice. Confidence is for the predicted class.

In [3]:
results_path = DATA / "rpt16_results.csv"
metadata_path = DATA / "rpt16_run_metadata.json"
if results_path.exists() and metadata_path.exists():
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    results = pd.read_csv(results_path)
    assert metadata["source"] == "btp_deployment"
    assert metadata["request_sha256"] == hashlib.sha256(request_path.read_bytes()).hexdigest()
    assert metadata["results_sha256"] == hashlib.sha256(results_path.read_bytes()).hexdigest()
    assert len(results) == PREDICTION_ROWS and results.invoice_id.is_unique
    assert results.invoice_id.tolist() == source.invoice_id.iloc[CONTEXT_ROWS:].tolist()
    metric_rows = []
    for target in TARGETS:
        actual = results[f"actual_{target}"]
        predicted = results[f"predicted_{target}"]
        confidence = results[f"confidence_{target}"]
        assert actual.tolist() == source[target].iloc[CONTEXT_ROWS:].tolist()
        assert predicted.isin([0, 1]).all() and confidence.between(0, 1).all()
        metric_rows.append({"Target": target, "Positive cases": int(actual.sum()),
                            "Accuracy": accuracy_score(actual, predicted),
                            "Precision": precision_score(actual, predicted, zero_division=0),
                            "Recall": recall_score(actual, predicted, zero_division=0),
                            "ROC AUC": "N/A (top-1)"})
    display(pd.DataFrame(metric_rows).set_index("Target").round(3))
    display(results[["invoice_id", "actual_paid_late", "predicted_paid_late", "confidence_paid_late",
                     "actual_is_fraud", "predicted_is_fraud", "confidence_is_fraud"]].head(8).round(3))
    display(pd.DataFrame([{key: metadata.get(key) for key in ("run_at_utc", "source", "model_configured", "test_rows")}]).T.rename(columns={0: "Value"}))
else:
    display(Markdown("**No direct BTP result is saved yet.** After a successful live run, the result and metadata will appear here."))

,Positive cases,Accuracy,Precision,Recall,ROC AUC
Target,,,,,
paid_late,51,0.711,0.630,0.667,N/A (top-1)
is_fraud,22,0.867,0.667,0.455,N/A (top-1)


,invoice_id,actual_paid_late,predicted_paid_late,confidence_paid_late,actual_is_fraud,predicted_is_fraud,confidence_is_fraud
0,SYN-01025,1,0,0.58,0,0,0.85
1,SYN-01026,0,0,0.60,1,0,0.54
2,SYN-01027,0,1,0.56,0,0,0.89
3,SYN-01028,0,0,0.54,0,0,0.91
4,SYN-01029,1,0,0.73,0,0,0.92
5,SYN-01030,0,0,0.78,0,0,0.92
6,SYN-01031,0,0,0.59,0,0,0.91
7,SYN-01032,0,0,0.81,0,0,0.92


,Value
run_at_utc,2026-10-03T22:17:05.746027+00:00
source,btp_deployment
model_configured,sap-rpt-1.6-large
test_rows,128


## Next step

Open `compare_saved_results.ipynb` to compare the saved direct BTP predictions with XGBoost. The comparison notebook makes no BTP call.